# 02 - Fine-tuning IndoBERT untuk sentimen (neg / neu / pos)

Bisa dijalankan **lokal** (kernel `.venv`, GPU NVIDIA dengan torch CUDA) atau di **Google Colab** (*Runtime → Change runtime type → T4 GPU*). Lalu *Run all*.
Lokal: notebook mencari `data/processed/dataset.csv` sendiri (naik dari folder notebook), dan `gold.csv` harus ada di folder yang sama dengan `dataset.csv`.

- **Masuk**: `dataset.csv` (dari `src/make_dataset.py`). Opsional: `gold.csv` (dari `src/make_gold.py --gabung`) untuk melihat skor di gold set.
- **Keluar**: folder `hasil_indobert/` (di dalam folder data; di Colab juga diunduh sebagai `hasil_indobert.zip`) berisi `pred_indobert_s*_test.csv` (prediksi test per seed) dan `ringkasan.json`.

**Protokol sama dengan baseline TF-IDF + LR:** model dan epoch terbaik dipilih memakai macro-F1 di set **val** (label bintang).
Set test dan gold set tidak dipakai untuk memilih apa pun; keduanya hanya dinilai setelah pelatihan selesai.
Pelatihan diulang dengan 3 seed supaya kita bisa melaporkan rata-rata ± simpangan, bukan satu angka keberuntungan.

In [1]:
import os, sys, json, math, random, time, zipfile, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("PERINGATAN: tidak ada GPU, pelatihan akan sangat lambat. Lokal: pasang torch versi CUDA. Colab: Runtime -> Change runtime type -> T4 GPU.")

torch 2.14.1+cu126 | device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU | VRAM total: 6.4 GB


In [ ]:
try:
    import transformers
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
print("transformers", transformers.__version__)

## Pengaturan

In [ ]:
MODEL = os.environ.get("MODEL_NAME", "indobenchmark/indobert-base-p1")
KOLOM_TEKS = "text"        # teks bersih: slang dinormalisasi, emoji jadi kata, TANPA stemming
MAX_LEN = 128
BATCH = int(os.environ.get("BATCH", 16))   # 16 aman untuk GPU 6 GB; naikkan ke 32 kalau VRAM lega
LR = float(os.environ.get("LR", 2e-5))
EPOCHS = int(os.environ.get("EPOCHS", 4))
WARMUP = 0.1
WEIGHT_DECAY = 0.01
SEEDS = json.loads(os.environ.get("SEEDS", "[42, 7, 2024]"))
SIMPAN_MODEL = False       # True: simpan model seed terbaik (menurut val) ke Google Drive, untuk demo nanti

def cari_data_dir():
    if os.environ.get("DATA_DIR"):
        return os.environ["DATA_DIR"]
    p = os.path.abspath(os.getcwd())
    for _ in range(4):                      # naik maksimal 4 level mencari data/processed/dataset.csv
        kandidat = os.path.join(p, "data", "processed")
        if os.path.exists(os.path.join(kandidat, "dataset.csv")):
            return kandidat
        p = os.path.dirname(p)
    return "."                              # Colab: berkas diunggah ke folder kerja

DATA_DIR = cari_data_dir()
OUT_DIR = os.environ.get("OUT_DIR", os.path.join(DATA_DIR, "hasil_indobert"))
print("DATA_DIR:", os.path.abspath(DATA_DIR))
NAMA = ["neg", "neu", "pos"]

## Data

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists(os.path.join(DATA_DIR, "dataset.csv")):
    from google.colab import files
    print("Pilih dataset.csv (dan gold.csv kalau ada), lalu tunggu sampai selesai diunggah.")
    files.upload()

df = pd.read_csv(os.path.join(DATA_DIR, "dataset.csv"), dtype={"reviewId": str}).fillna({KOLOM_TEKS: ""})
tr, va, te = (df[df["split"] == s].reset_index(drop=True) for s in ("train", "val", "test"))
y_tr, y_va, y_te = tr["label"].values, va["label"].values, te["label"].values
print(f"train {len(tr)} | val {len(va)} | test {len(te)}")
print("train per kelas:", dict(zip(NAMA, np.bincount(y_tr, minlength=3))))
assert (tr[KOLOM_TEKS].str.strip() != "").all() and (te[KOLOM_TEKS].str.strip() != "").all()

gold_path = os.path.join(DATA_DIR, "gold.csv")
gold = pd.read_csv(gold_path, dtype={"reviewId": str}) if os.path.exists(gold_path) else None
if gold is not None:
    assert set(gold["reviewId"]) <= set(te["reviewId"]), "gold harus bagian dari split test"
    assert not set(gold["reviewId"]) & set(tr["reviewId"]), "gold tidak boleh ada di train"
print("gold set:", "tidak ada (dilewati)" if gold is None else f"{len(gold)} ulasan")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
enc = {n: tok(list(d[KOLOM_TEKS]), truncation=True, max_length=MAX_LEN)
       for n, d in (("train", tr), ("val", va), ("test", te))}
panjang = np.array([len(x) for x in enc["train"]["input_ids"]])
print(f"panjang token (train): median {np.median(panjang):.0f} | persentil 95 {np.percentile(panjang, 95):.0f} "
      f"| terpotong di {MAX_LEN} token: {np.mean(panjang >= MAX_LEN):.1%}")

## Fungsi pelatihan
Loop pelatihan ditulis langsung (tanpa `Trainer`) supaya tidak rusak oleh perubahan versi `transformers`.

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def buat_batch(encd, idx, labels=None):
    fitur = [{k: encd[k][i] for k in encd.keys()} for i in idx]
    b = tok.pad(fitur, return_tensors="pt")
    if labels is not None:
        b["labels"] = torch.tensor([int(labels[i]) for i in idx])
    return b

@torch.no_grad()
def prediksi_proba(model, encd, batch=64):
    model.eval()
    n = len(encd["input_ids"])
    urut = sorted(range(n), key=lambda i: len(encd["input_ids"][i]))   # panjang serupa dikelompokkan: lebih cepat
    hasil = np.zeros((n, 3), dtype=np.float32)
    for a in range(0, n, batch):
        idx = urut[a:a + batch]
        b = {k: v.to(device) for k, v in buat_batch(encd, idx).items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logit = model(**b).logits
        hasil[idx] = torch.softmax(logit.float(), dim=-1).cpu().numpy()
    return hasil

def buat_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=use_amp)

# bobot kelas seimbang (sama dengan class_weight="balanced" pada baseline)
cnt = np.bincount(y_tr, minlength=3)
bobot = torch.tensor(len(y_tr) / (3 * cnt), dtype=torch.float32, device=device)

def latih(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3, ignore_mismatched_sizes=True).to(device)
    per_epoch = math.ceil(len(y_tr) / BATCH)
    total = per_epoch * EPOCHS
    tanpa_decay = ("bias", "LayerNorm.weight")
    grup = [
        {"params": [p for n, p in model.named_parameters() if not any(t in n for t in tanpa_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(t in n for t in tanpa_decay)], "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(grup, lr=LR)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = buat_scaler()
    loss_fn = nn.CrossEntropyLoss(weight=bobot)
    rng = np.random.RandomState(seed)
    terbaik, riwayat, t0 = {"f1": -1.0}, [], time.time()
    for ep in range(1, EPOCHS + 1):
        model.train()
        perm = rng.permutation(len(y_tr))
        jumlah_loss = 0.0
        for a in range(0, len(perm), BATCH):
            idx = perm[a:a + BATCH]
            b = {k: v.to(device) for k, v in buat_batch(enc["train"], idx, y_tr).items()}
            labels = b.pop("labels")
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logit = model(**b).logits
            loss = loss_fn(logit.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            jumlah_loss += loss.item() * len(idx)
        p_va = prediksi_proba(model, enc["val"])
        f1 = float(f1_score(y_va, p_va.argmax(1), average="macro"))
        riwayat.append({"epoch": ep, "loss_train": jumlah_loss / len(y_tr), "f1_val": f1,
                        "acc_val": float(accuracy_score(y_va, p_va.argmax(1)))})
        print(f"  seed {seed} | epoch {ep}/{EPOCHS} | loss {riwayat[-1]['loss_train']:.4f} | val macro-F1 {f1:.4f}")
        if f1 > terbaik["f1"]:
            terbaik = {"f1": f1, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
    model.load_state_dict(terbaik["state"])          # kembali ke epoch dengan val terbaik
    p_te = prediksi_proba(model, enc["test"])
    hasil = {"seed": seed, "epoch_terbaik": terbaik["epoch"], "f1_val": terbaik["f1"],
             "f1_test_bintang": float(f1_score(y_te, p_te.argmax(1), average="macro")),
             "acc_test_bintang": float(accuracy_score(y_te, p_te.argmax(1))),
             "menit": round((time.time() - t0) / 60, 1), "riwayat": riwayat}
    del model, opt
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return hasil, p_te, terbaik["state"]

## Latih (3 seed)
Perkiraan kasar (belum terukur): beberapa menit per seed di GPU kelas T4 / RTX 3060. Kalau muncul `CUDA out of memory`, turunkan `BATCH` atau `MAX_LEN` di sel pengaturan.

In [ ]:
ringkas, proba_test = [], {}
state_global, f1_global, seed_global = None, -1.0, None
for s in SEEDS:
    print(f"== seed {s} ==")
    hasil, p_te, st = latih(s)
    ringkas.append(hasil); proba_test[s] = p_te
    print(f"  -> epoch terbaik {hasil['epoch_terbaik']} | val {hasil['f1_val']:.4f} | test (label bintang) {hasil['f1_test_bintang']:.4f} | {hasil['menit']} menit")
    if hasil["f1_val"] > f1_global:
        f1_global, state_global, seed_global = hasil["f1_val"], st, s
    else:
        del st
print("\nseed terbaik menurut VAL:", seed_global)

## Ringkasan

In [ ]:
tabel = pd.DataFrame(ringkas).drop(columns="riwayat")
print(tabel.round(4).to_string(index=False))
def ms(x):
    return f"{np.mean(x):.3f} ± {np.std(x, ddof=1):.3f}" if len(x) > 1 else f"{np.mean(x):.3f}"
print(f"\nmacro-F1 val                 : {ms(tabel['f1_val'])}")
print(f"macro-F1 test (label bintang): {ms(tabel['f1_test_bintang'])}   (baseline TF-IDF + LR: 0.571)")

if gold is not None:
    g = te[["reviewId"]].assign(i=np.arange(len(te))).merge(gold[["reviewId", "label_gold"]], on="reviewId")
    yg = g["label_gold"].map({n: i for i, n in enumerate(NAMA)}).values
    f_gold = [f1_score(yg, proba_test[s][g["i"].values].argmax(1), average="macro", labels=[0, 1, 2]) for s in SEEDS]
    print(f"macro-F1 gold ({len(g)} ulasan)    : {ms(f_gold)}   (baseline TF-IDF + LR: 0.636)")

## Simpan hasil
Lokal: berkas tersimpan langsung di `data/processed/hasil_indobert/`. Colab: zip diunduh otomatis, ekstrak ke folder itu.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
for s in SEEDS:
    p = proba_test[s]
    pd.DataFrame({"reviewId": te["reviewId"], "label": y_te, "pred": p.argmax(1),
                  "p_neg": p[:, 0].round(5), "p_neu": p[:, 1].round(5), "p_pos": p[:, 2].round(5)}
                 ).to_csv(f"{OUT_DIR}/pred_indobert_s{s}_test.csv", index=False)
with open(f"{OUT_DIR}/ringkasan.json", "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "kolom_teks": KOLOM_TEKS, "max_len": MAX_LEN, "batch": BATCH, "lr": LR, "epochs": EPOCHS,
               "warmup": WARMUP, "weight_decay": WEIGHT_DECAY, "seeds": SEEDS, "seed_terbaik_val": seed_global,
               "n_train": len(tr), "n_val": len(va), "n_test": len(te), "hasil": ringkas}, f, indent=2)
print("tersimpan di:", os.path.abspath(OUT_DIR), sorted(os.listdir(OUT_DIR)))
if IN_COLAB:
    zip_path = "hasil_indobert.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for nama in sorted(os.listdir(OUT_DIR)):
            z.write(os.path.join(OUT_DIR, nama), arcname=nama)
    from google.colab import files
    files.download(zip_path)

## (Opsional) Simpan model terbaik ke Google Drive
Ubah `SIMPAN_MODEL = True` di sel pengaturan. Dibutuhkan nanti untuk demo Streamlit / Hugging Face Spaces. Ukurannya sekitar 450 MB.

In [ ]:
if SIMPAN_MODEL:
    from google.colab import drive
    drive.mount("/content/drive")
    tujuan = "/content/drive/MyDrive/paylater-indobert"
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3, ignore_mismatched_sizes=True)
    model.load_state_dict(state_global)
    model.config.id2label = {i: n for i, n in enumerate(NAMA)}
    model.config.label2id = {n: i for i, n in enumerate(NAMA)}
    model.save_pretrained(tujuan)
    tok.save_pretrained(tujuan)
    print("model seed", seed_global, "disimpan di", tujuan)
else:
    print("SIMPAN_MODEL = False: model tidak disimpan.")